# Entrenamiento de Modelos de Machine Learning

1. **Preprocesar los datos de forma científica** usando Pipelines y `ColumnTransformer` de Scikit-Learn (para evitar la fuga de datos del conjunto de prueba).
2. **Entrenar y ajustar 5 algoritmos específicos** (Regresión Logística, Árbol de Decisión, Random Forest, XGBoost y una Red Neuronal profunda con Keras/TensorFlow).
3. **Serializar y guardar** los modelos para producción.

In [ ]:
import sys
from pathlib import Path

import joblib
import pandas as pd

# Deep Learning
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

# Modelos
from sklearn.linear_model import LogisticRegression

# Importes de Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from tensorflow import keras
from xgboost import XGBClassifier

# Añadir directorio raíz al path para usar src
sys.path.append(str(Path.cwd().parent))

## 1. Cargar y Preprocesar los Datos

Primero dividimos los datos en **Entrenamiento (Train)** y **Prueba (Test)**. 

El conjunto de prueba representa el futuro de nuestro modelo. **Nunca** debemos preprocesar todo el dataset junto. Si calculamos el promedio de una columna sobre todo el dataset para rellenar nulos, el promedio del test afectaría al de train, causando una sutil "fuga de información" (*Data Leakage*). 

> **Solución**: Ajustaremos nuestro preprocesador (el `ColumnTransformer`) únicamente con los datos de entrenamiento (`fit_transform`) y lo aplicamos al de prueba únicamente para transformar (`transform`).

In [ ]:
from src.config import COLS_TO_DROP, MODELS_DIR, RAW_DATA_PATH, TARGET_COL

# 1. Cargar dataset
df = pd.read_csv(RAW_DATA_PATH)

# 2. Eliminar columnas irrelevantes o que causan fuga de datos
df_clean = df.drop(columns=[col for col in COLS_TO_DROP if col in df.columns])

# 3. Separar X e y
X = df_clean.drop(columns=[TARGET_COL])
y = df_clean[TARGET_COL]

# 4. Dividir de forma estratificada (mismo ratio de cancelaciones en ambos conjuntos)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Conjunto de Entrenamiento: {X_train.shape[0]:,} filas")
print(f"Conjunto de Prueba: {X_test.shape[0]:,} filas")

### Construcción del Preprocessor Modular
Usaremos Pipelines de Scikit-Learn:
* Para numéricas: rellenar vacíos con la mediana e imputar escala estándar (`StandardScaler`).
* Para categóricas: rellenar vacíos con "Unknown" y aplicar codificación binaria (`OneHotEncoder`).

In [ ]:
# Separar columnas por tipo
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

# Crear pipeline para variables numéricas
num_pipeline = Pipeline(
    [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

# Crear pipeline para variables categóricas
cat_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

# Combinar en un único ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[("num", num_pipeline, num_cols), ("cat", cat_pipeline, cat_cols)]
)

## 2. Entrenamiento de los 4 Modelos Clásicos

Entrenaremos los 4 algoritmos tradicionales encapsulándolos en un Pipeline que une el preprocesador y el modelo. Así, cuando les pasemos filas sin limpiar en el futuro, se encargarian del preprocesamiento internamente.

In [ ]:
# Diccionario de modelos clásicos con hiperparámetros estables
classical_models = {
    "logistic_regression": LogisticRegression(max_iter=1000, random_state=42),
    "decision_tree": DecisionTreeClassifier(max_depth=10, random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42),
    "xgboost": XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42),
}

trained_pipelines = {}

for name, model in classical_models.items():
    print(f"Entrenando pipeline para {name.replace('_', ' ').title()}...")

    # Crear pipeline completo
    pipeline = Pipeline([("preprocessor", preprocessor), ("classifier", model)])

    # Ajustar modelo con datos de train
    pipeline.fit(X_train, y_train)
    trained_pipelines[name] = pipeline

    # Guardar pipeline en un archivo pkl en models/
    joblib.dump(pipeline, MODELS_DIR / f"{name}.pkl")
    print(f"Guardado correctamente en models/{name}.pkl\n")

## 3. Entrenamiento de la Red Neuronal de Keras (Deep Learning)

Con las Redes Neuronales aprendemos patrones no lineales muy complejos. 
Como Keras no es un clasificador nativo de Scikit-Learn, preprocesaremos los datos primero usando nuestro ColumnTransformer de forma externa y luego entrenaremos el modelo.

In [ ]:
print("Preparando datos para la Red Neuronal...")

# Ajustamos el preprocesador y transformamos los datos a formato matricial
X_train_processed = preprocessor.fit_transform(X_train)
input_dim = X_train_processed.shape[1]
print(f"Dimensión del vector de entrada: {input_dim} columnas preprocesadas.")

In [ ]:
# Diseñar la arquitectura secuencial de la Red Neuronal profunda
def create_neural_network(input_dimension):
    model = keras.Sequential(
        [
            keras.layers.Input(shape=(input_dimension,)),
            keras.layers.Dense(32, activation="relu"),
            keras.layers.Dropout(0.2),  # Regularización para evitar sobreajuste
            keras.layers.Dense(16, activation="relu"),
            keras.layers.Dense(1, activation="sigmoid"),  # Sigmoide para predicción binaria (0 o 1)
        ]
    )

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",  # Pérdida estándar de clasificación binaria
        metrics=["accuracy"],
    )
    return model


keras_model = create_neural_network(input_dim)
keras_model.summary()

In [ ]:
print("Entrenando Red Neuronal Keras (5 épocas para prueba rápida)...")

history = keras_model.fit(
    X_train_processed, y_train, epochs=5, batch_size=128, validation_split=0.1, verbose=1
)

In [ ]:
print("Serializando componentes de la Red Neuronal...")
# Guardamos el preprocesador que usaremos antes de llamar a Keras
joblib.dump(preprocessor, MODELS_DIR / "nn_preprocessor.pkl")
# Guardamos el modelo en formato nativo de Keras (.keras)
keras_model.save(MODELS_DIR / "neural_network.keras")
print("¡Componentes de Deep Learning guardados con éxito!")